In [1]:
import os
import cv2

# === 設定路徑 ===
ori_base = r"C:\temp\classification_FINAL_ori_IMG"
yolo_txt_dir = r"C:\Users\howar\OneDrive\桌面\ALL_YOLO標記檔"
output_base = r"C:\temp\classification_FINAL_ROI"

# === 建立 ROI 資料夾結構 ===
splits = ['train', 'val', 'test']
classes = ['benign', 'malignant']

for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(output_base, split, cls), exist_ok=True)

# === 進行裁切並儲存 ===
for split in splits:
    for cls in classes:
        img_dir = os.path.join(ori_base, split, cls)
        out_dir = os.path.join(output_base, split, cls)

        for fname in os.listdir(img_dir):
            if not fname.lower().endswith('.png'):
                continue
            
            name, _ = os.path.splitext(fname)
            img_path = os.path.join(img_dir, fname)
            txt_path = os.path.join(yolo_txt_dir, name + '.txt')
            
            if not os.path.exists(txt_path):
                print(f"⚠️ 找不到標註檔: {txt_path}")
                continue

            # 讀取圖片與標註
            img = cv2.imread(img_path)
            h, w = img.shape[:2]

            with open(txt_path, 'r') as f:
                lines = f.readlines()
                for i, line in enumerate(lines):
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    _, x_center, y_center, box_w, box_h = map(float, parts)

                    # YOLO 格式轉為實際座標
                    x1 = int((x_center - box_w / 2) * w)
                    y1 = int((y_center - box_h / 2) * h)
                    x2 = int((x_center + box_w / 2) * w)
                    y2 = int((y_center + box_h / 2) * h)

                    # 邊界檢查
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(w - 1, x2), min(h - 1, y2)

                    roi = img[y1:y2, x1:x2]
                    if roi.size == 0:
                        print(f"❌ 空ROI: {img_path}")
                        continue

                    # 儲存圖像（若有多個框，則加上索引）
                    roi_name = f"[ROI]{name}.png" if len(lines) == 1 else f"[ROI]{name}_{i+1}.png"
                    cv2.imwrite(os.path.join(out_dir, roi_name), roi)

print("✅ 裁切完成並儲存至 classification_FINAL_ROI")


✅ 裁切完成並儲存至 classification_FINAL_ROI
